# Attention：从 SDPA 到 VarLen

Transformer 每一层都靠 attention 让序列里的 token 交换信息：每个位置的 query 与 key 计算相似度，再按 mask 允许的范围对 value 做加权求和。对本章的 causal decoder 来说，一个 token 只能读取当前及更早位置；document-aware mask 还会进一步限制它只能读取同一条样本。公式写作：

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{D}}+M\right)V$$

$D$ 是 head dimension，$M$ 是 attention mask。公式本身简洁——两步矩阵乘 $QK^\top$ 和 $PV$，中间夹一个 softmax——但全量 $QK^\top$ 有 $S\times S$ 个元素。序列长度 $S$ 一上去，计算量按 $O(S^2)$ 涨，完整 score 矩阵的显存开销也按 $O(S^2)$ 涨。这个 $O(S^2)$ 就是 attention 所有优化要面对的共同约束——不同技术从不同位置对它下手。

本章覆盖五种机制，每种改变的是 attention pipeline 里的不同环节。SDPA 是统一数学接口加 backend dispatch——调用同一个 API，实际跑了哪个 kernel 由 dispatcher 决定。FlashAttention 改的是 kernel 的执行方式：通过融合和 tiling 减少 HBM 往返，不改变 dense attention 的数值结果。Qwen3 的 GQA 则是一个独立的维度——把 K/V head 数从 16 砍到 8，缩小的是 K/V 投影和缓存，跟注意力矩阵的 token pair 数量无关。稀疏注意力走的是另一条路：在既定 mask 的允许域中，如果提前告诉 kernel 哪些 pair 可以不算，就能进一步减少实际计算量。SFT 的 document mask 把这个想法具体化为一条规则：同一个序列里的不同文档之间不能互相看到。最后，VarLen Attention 把 document mask 变成 kernel 能消费的累计长度，让 NPU 在计算前就知道可以跳过所有跨文档 tile。

05.03 会为这些机制逐项设计对照实验，用 wall-time 和 trace 验证。

## 1. SDPA：统一的 Attention 接口

公式在章首已经给出：$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}(QK^\top/\sqrt{D}+M)V$。以 PyTorch 常用的 BNSD 布局为例：

- $Q$：`[B, Nq, S, D]`；
- $K,V$：`[B, Nkv, S, D]`；
- 逻辑 score：`[B, Nq, S, S]`。

主计算在 $QK^\top$ 和 $PV$，随序列长度按 $O(BN_qS^2D)$ 增长。如果显式保存 score，中间张量是 $O(BN_qS^2)$——长序列下同时吃算力和显存。

PyTorch 用一个 API 收拢这些计算：

~~~python
torch.nn.functional.scaled_dot_product_attention(
    query, key, value,
    attn_mask=None,
    dropout_p=0.0,
    is_causal=True,
    enable_gqa=True,
)
~~~

SDPA 是**接口**，不是某一个固定 kernel。调用同一个 API，实际跑什么取决于 dispatcher 根据 device、dtype、shape、layout 和 mask 做的路由选择。对本课程锁定的 NPU 软件栈和输入——bf16、`[B, Nq/Nkv, S, 128]`、`is_causal=True`——`op-plugin` 中的 `ScaledDotProductAttentionKernelNpuOpApi.cpp` 会把 `is_causal` 映射为 `sparse_mode=2`，再调用 `aclnnFlashAttentionScore`。05.03 对这组输入采集的 trace 中确实出现了 `FlashAttentionScore`，而不是离散的 MatMul 和 Softmax；这条观察确认的是本节实际执行路径，换 shape、mask 或软件版本后仍应重新看 trace。

在本课程使用的 NPU fused 分支中，`is_causal=True` 时不能再传显式 `attn_mask`。document-aware block-causal 路径的做法是把因果关系也编码进布尔 allow mask，并以 `is_causal=False` 调用。不要把这条 NPU 路由限制外推成所有 PyTorch backend 的统一规则；不同 backend 和版本对两者组合的支持可能不同。

> **从 05.03 的 trace 可以确认**：本节固定输入实际命中了 FlashAttention，而不只是从 SDPA 的 API 名称猜测 backend。`enable_gqa=True` 则告诉 dispatcher Q 和 K/V 的 head 数可以不相等，避免调用方手动 `repeat_interleave`，并让 backend 有机会直接消费紧凑的 K/V 表示。

## 2. FlashAttention：减少 IO，不改变结果

最直接的 attention 实现依次启动：矩阵乘 → mask → softmax → dropout → 第二次矩阵乘。完整的 $QK^\top$ score 多次写入、读出 HBM，算子之间还有 launch 和同步开销。

FlashAttention 的核心是**融合 + 分块（tiling）+ online softmax**：

1. 把 Q、K、V 切成 tile；
2. 每次只搬当前 tile 到片上 SRAM；
3. 在片上完成局部 score、mask、softmax 统计量和 $PV$ 累加；
4. 只写最终输出到 HBM——不物化完整的 $S\times S$ score。

效果：

- **HBM 流量更少**：不反复读写大型中间张量；
- **峰值显存更低**：不保存完整 score；
- **算子碎片更少**：多个操作合并到一个 kernel。

FlashAttention **不改变 dense attention 的数值结果，也不改变 $O(S^2)$ 的计算复杂度**。它改善的是 IO 和硬件利用率。§4 的稀疏注意力和 §6 的 VarLen 才会进一步讨论"哪些 token pair 可以根本不算"。

> 参考：[FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135)

<figure>
  <picture>
    <source srcset="images/05.02_flash_attention_execution_flow.svg" type="image/svg+xml">
    <img src="images/05.02_flash_attention_execution_flow.png" alt="上下两条 Attention 执行流对比：eager 路径依次运行 QK、写回完整 S×S score、mask 与 softmax、再读回概率做 PV；FlashAttention 路径把 QKV tile 搬入片上 SRAM，在一个融合 tile loop 中完成 QK、mask、online softmax 和 PV 累加，只写最终输出。" loading="lazy" style="max-width: 100%; height: auto;">
  </picture>
  <figcaption>图 05.02-1：两条路径计算同一个 dense attention；FlashAttention 消除的是完整 score 的 HBM 往返和算子碎片，而不是 O(S<sup>2</sup>) 计算。</figcaption>
</figure>

## 3. Qwen3 的 GQA：缩小 K/V 侧

本课程使用的 Qwen3-1.7B 配置：$N_q=16$、$N_{kv}=8$、$D=128$。Query 有 16 个 head，Key/Value 各只有 8 个 head——每两个 query head 共享一组 K/V。这就是 Grouped-Query Attention（GQA）。

对比 $N_q=N_{kv}$ 的 Multi-Head Attention（MHA），GQA 的变化集中在 K/V 侧。K/V projection 的参数和计算量约为 MHA 的 50%——只生成 8 个 KV head 而非 16 个。K/V activation 和推理时的 KV cache 同理，存储量与 $N_{kv}$ 成正比。

但主计算量不受影响。逻辑 score 仍然是 `[B,16,S,S]`——score 由 query head 数 $N_q=16$ 和序列长度决定，跟 K/V head 数无关。$QK^\top$ 和 $PV$ 这两个矩阵乘的计算量也由 $N_q$ 驱动，不减半。

05.03 的固定-shape 对照保持 `B=2,S=4096,Nq=16` 不变，只切换 $N_{kv}$。从这组实验可以观察到：K/V tensor 从 `64.0 MiB` 精确减半到 `32.0 MiB`，逻辑 score 保持 `1024.0 MiB`；但 forward 只从 `1.581 ms` 降到 `1.482 ms`，fwd+bwd 也只快了 8.7%。这些测量点说明，在当前 backend 和 shape 上，K/V 字节减半没有转化成整个 SDPA 算子的 2× 加速。它们不定义其他 shape 或 backend 的固定加速比例；实际 wall-time 还取决于 kernel 如何实现 GQA，以及瓶颈落在计算、搬运还是调度。

PyTorch SDPA 通过 `enable_gqa=True` 直接接收 $N_q \neq N_{kv}$ 的 Q/K/V——调用方不用手动 `repeat_interleave`。backend 有机会直接消费紧凑的 K/V 表示，但是否命中原生 GQA kernel 要看 trace。

**可以直接推出的是**：`Nkv=16→8` 会把 K/V projection、activation 和 KV cache 的对应部分减半，而由 $N_q$ 决定的逻辑 score 与主矩阵乘不会随之减半。至于 SDPA wall-time 改善多少，应像 05.03 一样通过具体 workload 测量，不能由 head 数比例直接换算。

In [ ]:
B, S, NQ, NKV, D = 2, 4096, 16, 8, 128
BYTES_PER_ELEMENT = 2  # BF16 / FP16

def mib(elements: int) -> float:
    return elements * BYTES_PER_ELEMENT / 1024**2

kv_gqa = 2 * B * S * NKV * D
kv_mha = 2 * B * S * NQ * D
logical_scores = B * NQ * S * S

print(f'K+V with Qwen3 GQA: {mib(kv_gqa):.1f} MiB')
print(f'K+V with MHA:       {mib(kv_mha):.1f} MiB')
print(f'Logical scores:     {mib(logical_scores):.1f} MiB')
print(f'GQA KV reduction:   {1 - NKV / NQ:.1%}')
print('Score reduction:    0.0% (still indexed by query heads)')

## 4. 稀疏注意力：少算，不只是少搬

§2 的 FlashAttention 把 attention 分块融合，§3 的 GQA 在 head 维上缩小了 K/V 侧。普通 causal FlashAttention 可以利用下三角结构，不必处理被 causal mask 排除的整个上三角；但它不知道 packed 序列里的文档边界，因此仍会保留 causal 可见域内所有跨文档 pair，计算复杂度仍是 $O(S^2)$。GQA 改变的是 head 数，不改变每个 head 内的 token pair 可见性。

这引出了一个独立于 IO 优化和模型架构的方向：**稀疏注意力**——如果 kernel 提前知道某些 token pair 永远不可见，那就不需要计算它们。

稀疏注意力的"稀疏"指的不是矩阵里有很多零，而是**哪些 pair 需要算、哪些可以跳过，由一个事先确定的 pattern 来定义**。这个 pattern 被编码成 mask，kernel 在分块执行时可以跳过整块 mask 为空的 tile——不是算出来再扔掉，是从源头就不算。概念上，FlashAttention 主要回答"保留下来的 pair 怎么通过分块和融合算得更高效"，稀疏 pattern 回答"哪些 pair 不必算"；具体 kernel 可以同时实现两者，FlashAttention 的 VarLen 变体就是这种组合。

稀疏 pattern 要实用，必须满足一个条件：**硬件能利用它**。如果跳过的位置散落在矩阵各处、不成整块，GPU/NPU 就无法跳过整块 tile，实际 wall-time 不会有改善。因此实际可用的稀疏注意力几乎都是**结构化稀疏**——跳过的区域形成规整的 tile。常见的结构化 pattern 包括：causal mask（跳过上三角 tile）、sliding window（每个 query 只看最近的 $W$ 个 key，跳过窗口外的 tile）、block-sparse（注意力矩阵按固定大小的 block 划分，只计算选中的 block）、以及本课程最关心的 **VarLen / document-aware**——每个文档在自己的区间里做 causal attention，跨文档的整块区域全部跳过。

这正是 SFT packing 场景下的核心收益来源。当多个文档被打包进同一个序列时，只用 causal mask 会让后一个文档的 token 看到前一个文档——这既不符合"每个文档独立训练"的语义，也白白计算了大量跨文档 pair。把文档边界编码成稀疏 pattern（block-causal mask 或 VarLen 的 `cu_seq`），kernel 就可以跳过跨文档区域，减少的不只是 IO，也是实际的计算量。当前 TorchTitan 路径会把 padding 隔成单独区间，但不会在进入 Q/K/V 前压紧并删除 padding token；padding 区间内部的 causal pair 仍会计算。

下面 §5 定义 attention mask 在 SFT 场景下的具体语义——哪些 pair 必须隔离。§6 的 VarLen 则把这个文档感知的稀疏 pattern 直接交给 NPU kernel。

## 5. Attention Mask：SFT 为什么不止 causal

Attention mask 决定每个 query token 能看到哪些 key/value。自回归模型的基础约束是 causal：位置 $i$ 只能看到 $j\le i$。SDPA 用 `is_causal=True` 传递这个结构，不需要 $S\times S$ 的 dense mask。

### SFT 的额外需求

SFT 把多段对话或多个文档 packing 到同一个固定长度序列中。此时仅有 causal 不够：后一个文档的 token 虽然看不到未来，但仍可能看到前一个文档——这不符合"每个文档独立训练"的语义。正确的 document-aware 可见性是：

$$
M_{ij}=
\begin{cases}
0, & \operatorname{doc}(i)=\operatorname{doc}(j)\ \text{且}\ j\le i\\
-\infty, & \text{其他}
\end{cases}
$$

那么，现有的机制能不能替代 document mask？逐一来看。

`is_causal=True` 只约束时间维——位置 $i$ 不能看到 $j>i$——但它不关心位置 $i$ 和 $j$ 是否属于同一个文档。只要 $j \le i$，即使 $j$ 在上一个文档、甚至上上个文档，causal mask 照样放行。`labels == IGNORE_INDEX` 更帮不上忙——labels 是传给 loss 函数的，`F.scaled_dot_product_attention` 根本不接收 labels 参数，attention 算子的输入只有 Q、K、V 和 mask，labels 写到什么都不影响 attention 输出。padding mask 同理，它只管哪些位置是补齐的，不知道也不关心文档归属。**唯一能阻止跨文档 attention 的是把文档边界编码进 attention mask 本身**——dense 的 block-causal mask，或者 VarLen 的 `cu_seq`。

prompt、EOS 或 padding 的 label 被写成 `IGNORE_INDEX`，只代表 loss 不监督这些位置；它们的 K/V 仍会参与 attention。Packing 时，dataloader 会让每条样本的位置编号重新从 0 开始；trainer 看到编号再次变成 0，就知道一条新样本开始了，再把这些起点转成 attention metadata。EOS 只是消息结束符，不能承担这个职责。05.04 和 05.05 会追踪完整链路。

## 6. VarLen Attention：把边界提前交给 kernel

### 6.1 结构化稀疏：不算已知的无效 pair

§4 的稀疏注意力讲的是概念——用结构化的 mask pattern 告诉 kernel 哪些 pair 可以跳过。现在看它在 NPU 上的具体落地。

FlashAttention 减少了 IO；causal kernel 还可以利用已知的三角结构。但对 packed 序列，如果没有文档边界 metadata，它仍会计算 causal 可见域内的跨文档 pair。kernel 在计算前知道更多结构化边界后，才可以继续跳过对应的 tile。

不同的 mask 结构对应不同的跳过策略。普通的 causal mask 让 kernel 知道下三角关系，可以跳过上三角 tile。sliding window 或 block sparse 让 kernel 知道窗口大小或 block pattern，跳过窗口外的 tile。而 SFT 的 VarLen 给 kernel 的是每个区间的累计长度——kernel 由此知道 token `[0:L₁)` 是区间 0、`[L₁:L₁+L₂)` 是区间 1，区间之间互不可见，于是可以跳过跨区间区域。在当前 TorchTitan 实现中，真实样本和末尾 padding 都由这种区间边界描述；边界负责隔离 padding，却不会把 padding token 从 Q/K/V 中移除。

### 6.2 VarLen：用累计长度描述文档边界

Packing 后有 $m$ 个文档，长度 $L_1,\ldots,L_m$，真实 token 总数 $T=\sum_i L_i$。VarLen 把 Q/K/V 沿 token 维拼成 TND 布局：

- Q：`[T, Nq, D]`
- K/V：`[T, Nkv, D]`
- `cu_seq = [0, L_1, L_1+L_2, \ldots, T]` — 每个区间的半开边界

例如三个文档长度 `[3, 3, 2]` → `cu_seq = [0, 3, 6, 8]`，分别占 `[0:3)`、`[3:6)`、`[6:8)`。kernel 在每个区间内部做 causal attention，区间之间不计算。

> TorchTitan 保留含开头 0 的 `cu_seq` 格式；传给 CANN FA v3 时去掉开头的 0，变成累计结束位置 `[3, 6, 8]`。两者是同一个信息，只是格式不同。05.03 的实验和 05.04 的 `VarlenMetadata` 都遵循这个约定。

### 6.3 理论收益：VarLen 到底少算多少？

设一个 packed 序列包含 $m$ 个文档，长度为 $L_1,\ldots,L_m$，真实 token 总数 $T=\sum_i L_i$。只比较 causal attention 中实际需要计算的 query-key pair：

$$
C_{\text{packed}}=\frac{T(T+1)}{2}
$$

把所有 token 当成一个连续文档时，需要计算整个下三角。VarLen 按文档切开，只计算每个文档自己的下三角：

$$
C_{\text{varlen}}=\sum_{i=1}^{m}\frac{L_i(L_i+1)}{2}
$$

两者之差正好是所有跨文档 pair：

$$
C_{\text{skipped}}=C_{\text{packed}}-C_{\text{varlen}}=\sum_{i<j}L_iL_j
$$

> **明确的理论结论**
>
> 1. VarLen 精确跳过全部跨文档 causal pair；文档内应该计算的 pair 一个不少。
> 2. 固定 $T$ 和文档数 $m$ 时，文档越均匀，$C_{\text{varlen}}$ 越小，理论收益越大。等长文档 $L=T/m$ 给出最大收益。
> 3. 等长时，pair 比例的精确值为 $\frac{C_{\text{varlen}}}{C_{\text{packed}}}=\frac{T/m+1}{T+1}$；当 $T$ 很大时才近似为 $1/m$。因此逻辑工作量最多约减少到原来的 $1/m$，理论 pair speedup 最多约为 $m$ 倍。
> 4. 这个 $m$ 倍只是 **pair 数量的上限**，不是 wall-time 保证。tile 粒度、短文档、layout 转换、kernel launch 和带宽都会让实际加速低于它。

以 `T=4096`、`8×512` 为例：$C_{\text{packed}}=8,390,656$，$C_{\text{varlen}}=1,050,624$，精确跳过 $7,340,032$ 个 pair，即 `87.48%`；剩余 pair 比例为 `12.52%`，对应的理论 pair speedup 为 `7.99×`，而不是笼统的 `8× wall-time`。

上面的例子与 05.03 都令 `T=4096` 个位置全部属于真实文档，因此只比较跨文档 pair。当前 TorchTitan 训练 batch 若含末尾 padding，会把 padding 作为额外区间传给 VarLen；跨区间 pair 被隔离，但 padding 区间自身仍贡献 $L_{pad}(L_{pad}+1)/2$ 个 causal pair。只有在生成 Q/K/V 前先压紧 token，才算真正移除了 padding；06.05 会把这项单独计入。

### 6.4 NPU 上的具体调用

当前环境 `torch_npu 2.12.0rc1` 的 VarLen 训练路径使用 `torch.ops.npu.npu_fusion_attention_v3`。调用时 layout 设为 `"TND"`（token-major 拼接布局），`actual_seq_qlen` 和 `actual_seq_kvlen` 传 CPU 侧的 `int64` 累计结束位置（不含开头 0，即 `cu_seq[1:]`），`sparse_mode=7` 表示逐文档 causal VarLen，`head_num` 直接用 query head 数（GQA 下可大于 KV head 数）。

> **注意**：`torch_npu` 没有独立的 `varlen` 公开函数。NPU VarLen 就是 FA v3 + `sparse_mode=7`。05.05 的 `NPUVarlenAttention` 封装了这个调用。

<figure>
  <picture>
    <source srcset="images/05.02_varlen_metadata_flow.svg" type="image/svg+xml">
    <img src="images/05.02_varlen_metadata_flow.png" alt="VarLen 的样本区间与张量布局汇合流程：packed 样本 AAA、BBB、CC 的长度 3、3、2 变成 cu_seq 0、3、6、8 和 actual_seq 3、6、8；QKV 同时从 BSND reshape 为 TND，两路信息汇入 FusionAttention V3 的 TND sparse_mode 7 路径。" loading="lazy" style="max-width: 100%; height: auto;">
  </picture>
  <figcaption>图 05.02-2：样本区间与 TND 张量在 FA v3 汇合；同一份区间信息既隔离不同样本，也让 kernel 跳过样本之间的计算。</figcaption>
</figure>

In [ ]:
import torch


def torch_npu_varlen_attention(q_tnd, k_tnd, v_tnd, cu_seq, causal_mask):
    """用 CANN FA v3 执行逐文档 causal VarLen Attention。"""
    import torch_npu

    if q_tnd.ndim != 3 or k_tnd.ndim != 3 or v_tnd.ndim != 3:
        raise ValueError("Q/K/V 必须使用 [T, N, D] 的 TND 布局")
    if not (q_tnd.size(0) == k_tnd.size(0) == v_tnd.size(0)):
        raise ValueError("Q/K/V 的 token 总数 T 必须相同")

    cu_seq_cpu = cu_seq.to(device="cpu", dtype=torch.int64)
    if cu_seq_cpu.ndim != 1 or cu_seq_cpu.numel() < 2 or cu_seq_cpu[0].item() != 0:
        raise ValueError("cu_seq 必须是一维累计长度，并以 0 开头")
    if cu_seq_cpu[-1].item() != q_tnd.size(0):
        raise ValueError("cu_seq 的最后一个值必须等于 token 总数 T")

    seq_lengths = cu_seq_cpu[1:] - cu_seq_cpu[:-1]
    max_seq_len = int(seq_lengths.max().item())
    actual_seq = cu_seq_cpu[1:]

    return torch_npu.npu_fusion_attention_v3(
        query=q_tnd,
        key=k_tnd,
        value=v_tnd,
        head_num=q_tnd.size(1),
        input_layout="TND",
        atten_mask=causal_mask,
        scale=q_tnd.size(-1) ** -0.5,
        keep_prob=1.0,
        pre_tockens=max_seq_len,
        next_tockens=0,
        actual_seq_qlen=actual_seq,
        actual_seq_kvlen=actual_seq,
        sparse_mode=7,
    )[0]


# CPU 上先看清楚 metadata；真正的 Q/K/V 和 causal_mask 要放在 NPU 上。
doc_lengths = torch.tensor([3, 3, 2], dtype=torch.int64)
cu_seq = torch.cat([torch.zeros(1, dtype=torch.int64), doc_lengths.cumsum(0)])
total_tokens = int(cu_seq[-1].item())

dense_causal_pairs = total_tokens * (total_tokens + 1) // 2
varlen_causal_pairs = int((doc_lengths * (doc_lengths + 1) // 2).sum().item())

print("document lengths:", doc_lengths.tolist())
print("cu_seq kept by TorchTitan:", cu_seq.tolist())
print("actual_seq passed to FA v3:", cu_seq[1:].tolist())
print("Q / K / V TND shapes:", (total_tokens, 16, 128), (total_tokens, 8, 128), (total_tokens, 8, 128))
skipped_fraction = 1 - varlen_causal_pairs / dense_causal_pairs
print("logical causal pairs, concatenated / varlen:", dense_causal_pairs, "/", varlen_causal_pairs)
print(f"theoretical skipped fraction: {skipped_fraction:.1%}")


### 总结

回到开头的框架，现在可以看清楚每种技术到底改变了什么、没改变什么。

SDPA 统一了接口和 dispatch，但不决定实际跑哪个 kernel——看代码调用只说明走了统一入口。FlashAttention 通过融合和 tiling 改变了 kernel 的执行方式，减少 HBM 往返和算子碎片；普通 causal 版本可利用三角结构，但不知道 packed 文档边界，整体复杂度仍是 $O(S^2)$。GQA 缩小了 K/V 侧的参数、activation 和推理 cache；这是 head 维上的缩减，不改变 token pair 的可见性。结构化稀疏从另一个维度定义"哪些不用算"，具体 kernel 可以把这种边界跳过与 FlashAttention 的分块融合组合起来，VarLen 就是一例。SFT 的 document mask 在语义层面隔离文档——causal 和时间有关、和文档归属无关，labels 则根本不进 attention 算子。VarLen Attention 把文档边界变成 `cu_seq`，让 kernel 跳过跨文档区域；当前 TorchTitan 还会把 padding 隔成独立区间，而不是从 Q/K/V 中删除它。理论 pair 比例也不能直接当 wall-time 加速比，tile、launch、带宽等固定成本仍然存在。

05.03 用固定长度与多文档输入从算子层检验这些机制；05.04 生成真实 SFT 文档 metadata；05.05 追踪 `NPUVarlenAttention` 如何把它交给 FA v3。

## 练习

1. （判断题）只要代码调用 `F.scaled_dot_product_attention`，无需 trace 就能断定任意 shape、mask 和软件版本都命中同一个 FlashAttention kernel。

2. （判断题）FlashAttention 通过融合和 tiling 将 attention 的计算复杂度从 $O(S^2)$ 降到了 $O(S)$。

3. （单选题）Qwen3-1.7B 使用 GQA 将 $N_{kv}$ 从 16 减到 8。以下哪项不受此变化影响？
    A. K/V projection 的参数量
    B. 推理时的 KV cache 大小
    C. 逻辑 score 矩阵 `[B, Nq, S, S]` 的大小
    D. SDPA 收到的 K/V tensor 字节数

4. （判断题）packed SFT 中，把 prompt 和 padding 位置的 label 设为 `IGNORE_INDEX` 即可阻止跨文档 attention。

5. （单选题）VarLen 跳过跨文档计算时，传给 CANN FA v3 的关键参数组合是什么？
    A. `input_layout="BSND"` + `sparse_mode=0`
    B. `is_causal=True` + `enable_gqa=True`
    C. `input_layout="TND"` + `actual_seq` 累计长度 + `sparse_mode=7`
    D. `attn_mask` 为全零 bool tensor

6. （单选题）FlashAttention 和结构化稀疏的关系是？
    A. 二者只能选其一
    B. 任何 FlashAttention kernel 都会自动识别 packed 文档边界
    C. 前者主要优化保留 pair 的执行与 IO，后者定义可跳过的 pair；具体 kernel 可以同时实现两者
    D. 结构化稀疏只适用于推理

In [ ]:
!cat ./answer/05.02_answer.txt
